Load the dataset

In [1]:
import pandas as pd
from rags import RAG
rag = RAG()
meta_qa = pd.read_parquet("parquet_data/qa_nl.parquet")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import time

def call_rag_with_retry(query, max_retries=3, base_delay=2):
    for attempt in range(max_retries):
        try:
            return rag.JudgeRAG(query)
        except Exception as e:
            if attempt == max_retries - 1:
                # give up after final attempt, log and move on
                return {"answer": None, "error": str(e)}
            wait = base_delay * (2 ** attempt)  # exponential backoff: 2s, 4s, 8s...
            print(f"Retry {attempt+1}/{max_retries} after error: {e}. Waiting {wait}s...")
            time.sleep(wait)

In [ ]:
import pandas as pd
import time
from tqdm import tqdm

CHECKPOINT_EVERY = 100
results = []

overall_start = time.perf_counter()

for i, row in enumerate(tqdm(meta_qa.itertuples(index=False), total=len(meta_qa))):
    start = time.perf_counter()
    art_ids, answer = call_rag_with_retry(row.question)
    elapsed = time.perf_counter() - start

    results.append({
        "query_id": row.id,
        "query": row.question,
        "answer": answer,
        "retrieved_ids": art_ids,
        "latency_sec": elapsed
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_parquet(f"output/judgeRAG/outputs_partial{i+1}.parquet", index=False)

total_elapsed = time.perf_counter() - overall_start
print(f"Total time: {total_elapsed:.2f}s | Avg per query: {total_elapsed/len(meta_qa):.2f}s")

pd.DataFrame(results).to_parquet("output/judgeRAG/rag_outputs.parquet", index=False)

  0%|          | 0/2 [00:00<?, ?it/s]











100%|██████████| 2/2 [01:50<00:00, 55.06s/it]

Total time: 110.17s | Avg per query: 0.08s


In [5]:
outp = pd.read_parquet("output/judgeRAG/rag_outputs.parquet")
outp

,query_id,query,answer,retrieved_ids,latency_sec
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,Kort antwoord: een dagvaarding is een officiee...,"[4788, 9380, 4423, 9491, 15044, 14698, 14695, ...",86.252530
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Kort antwoord: het ouderlijk gezag wordt in pr...,"[3790, 3791, 3794, 3812, 3797, 3811, 3796, 393...",23.847746
